# model/01 — Spatial Baseline

Trains the Spatial Baseline model using only geographic co-occurrence features.

**Feature vector:** `[Vf (15D), Vp (15D), N (1D)]` = 31D

- **Vf**: PCA 15D embedding of plant binary existence matrix F
- **Vp**: PCA 15D embedding of pollinator binary existence matrix P
- **N**: count of 0.5° bins shared between plant and pollinator

No temporal signal of any kind. This is the baseline that all ANTHEIA
models are compared against.

**Single seed run (seed 42).** See `evaluation/01_seed_testing.ipynb`
for 5-seed results.

**Expected result:** ROC-AUC ≈ 0.957, PR-AUC ≈ 0.939

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from pathlib import Path

BASE    = Path("/scratch/ariana.l")
OLD_S4  = BASE / "Stage 4 Link Prediction Model"
NEW_S4  = BASE / "New Stage 4 Link Prediction Model"

VF_PATH = OLD_S4 / "stage4_Vf_phenofield.csv"
F_PATH  = OLD_S4 / "stage4_F_existence_phenofield.csv"
VP_PATH = NEW_S4 / "stage4_Vp_corrected.csv"
P_PATH  = NEW_S4 / "stage4_P_existence_corrected.csv"
GLOBI_PATH = OLD_S4 / "stage4_globi_conus_broad.csv"

SEED      = 42
NEG_RATIO = 3

print("Paths OK")

In [ ]:
# Load embeddings and existence matrices
print("Loading embeddings...")
Vf_df = pd.read_csv(VF_PATH, index_col=0)
Vp_df = pd.read_csv(VP_PATH, index_col=0)
F_df  = pd.read_csv(F_PATH,  index_col=0)
P_df  = pd.read_csv(P_PATH,  index_col=0)

print(f"  Vf : {Vf_df.shape}")
print(f"  Vp : {Vp_df.shape}")

# Common bins — must match between F and P
common_bins = [b for b in F_df.columns if b in set(P_df.columns)]
print(f"  Common bins: {len(common_bins)}")

F_common = F_df[common_bins].values
P_common = P_df[common_bins].values
fc_idx   = {sp: i for i, sp in enumerate(F_df.index)}
pc_idx   = {sp: i for i, sp in enumerate(P_df.index)}

def compute_N(plant, pollinator):
    """Shared bin count — always use F_common @ P_common, never raw F/P."""
    return float(F_common[fc_idx[plant]] @ P_common[pc_idx[pollinator]])

In [ ]:
# Load GloBI and build positive pairs
globi = pd.read_csv(GLOBI_PATH)
globi = globi.rename(columns={
    "sourceTaxonName": "plant_species",
    "targetTaxonName": "pollinator_species"
})
globi = globi.dropna(subset=["plant_species", "pollinator_species"])
globi = globi[["plant_species", "pollinator_species"]].drop_duplicates()

# Filter to species present in Vf and Vp
globi = globi[
    globi["plant_species"].isin(Vf_df.index) &
    globi["pollinator_species"].isin(Vp_df.index)
].reset_index(drop=True)

print(f"Positive pairs: {len(globi)}")

In [ ]:
# Sample negatives
rng = np.random.default_rng(SEED)
positive_set = set(zip(globi["plant_species"], globi["pollinator_species"]))
plant_list   = sorted(Vf_df.index)
poll_list    = sorted(Vp_df.index)
n_neg        = len(globi) * NEG_RATIO

negatives = []
while len(negatives) < n_neg:
    pl_sample = rng.choice(plant_list, size=n_neg * 2)
    po_sample = rng.choice(poll_list,  size=n_neg * 2)
    for pl, po in zip(pl_sample, po_sample):
        if (pl, po) not in positive_set:
            negatives.append((pl, po))
        if len(negatives) >= n_neg:
            break

neg_df = pd.DataFrame(negatives, columns=["plant_species", "pollinator_species"])
pairs  = pd.concat([
    globi.assign(label=1),
    neg_df.assign(label=0)
], ignore_index=True)

print(f"Total pairs: {len(pairs)} ({pairs['label'].sum()} positive, {(pairs['label']==0).sum()} negative)")

In [ ]:
# Assemble feature matrix: [Vf, Vp, N] = 31D
print("Assembling feature matrix...")
rows = []
for _, row in pairs.iterrows():
    pl, po = row["plant_species"], row["pollinator_species"]
    vf = Vf_df.loc[pl].values   # 15D
    vp = Vp_df.loc[po].values   # 15D
    n  = compute_N(pl, po)      # 1D
    rows.append(np.concatenate([vf, vp, [n]]))

X = np.array(rows)
y = pairs["label"].values
print(f"X shape: {X.shape}  (31D = 15D Vf + 15D Vp + 1D N)")

In [ ]:
# Train/test split and logistic regression
X_idx = np.arange(len(pairs))
train_idx, test_idx = train_test_split(
    X_idx, test_size=0.2, random_state=SEED, stratify=y
)

clf = LogisticRegression(max_iter=1000, random_state=SEED)
clf.fit(X[train_idx], y[train_idx])
y_prob = clf.predict_proba(X[test_idx])[:, 1]

roc = roc_auc_score(y[test_idx], y_prob)
pr  = average_precision_score(y[test_idx], y_prob)

print(f"Spatial Baseline (31D, seed {SEED}):")
print(f"  ROC-AUC : {roc:.4f}")
print(f"  PR-AUC  : {pr:.4f}")
print()
print("No temporal signal. All performance comes from spatial co-occurrence.")
print("ANTHEIA models add PPE plant-side temporal signal on top of this.")